# Initialization

In [15]:
# Install libraries
!pip install --quiet numpy librosa soundfile pandas tqdm datasets[audio] openai-whisper

In [16]:
# Imports
import os
import librosa
import random
import numpy as np
import pandas as pd
import soundfile as sf
import IPython.display as ipd

from tqdm import tqdm
from datasets import load_from_disk, Audio as HFAudio

In [17]:
# Mount drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
# Import preprocessing functions
import sys
sys.path.insert(0, '/content/drive/MyDrive/Target TTS Project/TargetTTS')

from src.preprocess import preprocess_audio, build_waxal_cache, TARGET_SAMPLE_RATE

In [19]:
# Configuration
BASE_PATH        = "/content/drive/MyDrive/Target TTS Project/TargetTTS"
TARGET_SR        = 16000
FADE_DURATION_MS = 10
FADE_SAMPLES     = int((FADE_DURATION_MS / 1000) * TARGET_SR)

# LibriSpeech
ENG_CSV         = f"{BASE_PATH}/data/mixture_recipes/librispeech_mixture_recipes.csv"
ENG_OUTPUT_DIR  = f"{BASE_PATH}/data/synthetic_mixtures/librispeech_synthetic_mixtures"

# WAXAL
TWI_CSV         = f"{BASE_PATH}/data/mixture_recipes/waxal_mixture_recipes.csv"
TWI_OUTPUT_DIR  = f"{BASE_PATH}/data/synthetic_mixtures/waxal_synthetic_mixtures"
WAXAL_DISK_PATH = f"{BASE_PATH}/data/waxal"

os.makedirs(ENG_OUTPUT_DIR, exist_ok=True)
os.makedirs(TWI_OUTPUT_DIR, exist_ok=True)

# English: Librispeech Dataset Audio Processing

In [20]:
# Core Mixing Functions
def get_active_rms(y, top_db=30):
    """Calculates RMS energy only on the non-silent parts of the audio."""
    # librosa.effects.split returns start/end intervals of active speech
    intervals = librosa.effects.split(y, top_db=top_db)
    if len(intervals) == 0:
        return np.sqrt(np.mean(y**2)) + 1e-8

    # Concatenate all active segments
    active = np.concatenate([y[s:e] for s, e in intervals])

    # Calculate RMS of active segments
    rms = np.sqrt(np.mean(active**2))
    return rms + 1e-8 # Add epsilon to prevent divide-by-zero


def apply_fade(snippet):
    """Applies a Hann window fade-in and fade-out to prevent audio clicks."""
    if len(snippet) < FADE_SAMPLES * 2:
        return snippet  # Too short to fade properly

    window = np.hanning(FADE_SAMPLES * 2)
    fade_in = window[:FADE_SAMPLES]
    fade_out = window[FADE_SAMPLES:]

    snippet_faded = snippet.copy()
    snippet_faded[:FADE_SAMPLES] *= fade_in
    snippet_faded[-FADE_SAMPLES:] *= fade_out
    return snippet_faded


def apply_energy_scaling(y_t, y_n, sir_db):
    """
    Rescales the noise waveform to achieve the target SIR
    based on the active RMS energy of both speakers.
    """
    # Energy scaling to hit target SIR
    rms_t = get_active_rms(y_t)
    rms_n = get_active_rms(y_n)

    # Calculate required noise RMS to hit the target SIR
    # SIR = 20 * log10(RMS_target / RMS_noise)
    scale = (rms_t / (10 ** (sir_db / 20))) / rms_n
    y_n_scaled = y_n * scale
    return y_n_scaled


def mix_audio_arrays(y_t, y_n, sir_db, overlap_ratio, output_path=None):
    """
    Mix target and noise arrays at a given SIR and overlap ratio.
    Saves to output_path if provided. Returns the mixed array.
    Returns None if overlap_ratio == 0.0 (clean baseline, no mixing needed).
    """
    if overlap_ratio == 0.0:
        return None

    # Energy scaling to hit target SIR
    y_n_scaled = apply_energy_scaling(y_t, y_n, sir_db)

    # Calculate total samples of overlap needed
    len_t, len_n = len(y_t), len(y_n_scaled)
    target_overlap_samples = int(len_t * overlap_ratio)

    # Break noise into 2 to 4 random snippets
    num_snippets   = random.randint(2, 4)
    snippet_length = target_overlap_samples // num_snippets

    # Place snippets randomly across the target
    # Divide the target into chunks to prevent snippets from overlapping each other
    chunk_size     = len_t // num_snippets

    y_mix = y_t.copy()
    noise_pointer = 0

    for i in range(num_snippets):
        # If we run out of noise audio, we loop back to the start
        if noise_pointer + snippet_length > len_n:
            noise_pointer = 0 # loop the noise

        # Need a snippet of length 'snippet_length' from the noise audio
        snippet = y_n_scaled[noise_pointer : noise_pointer + snippet_length]
        noise_pointer += snippet_length

        # Apply fades to prevent transient clicks
        snippet = apply_fade(snippet)

        # Pick a random start position within this chunk
        max_start = chunk_size - len(snippet)
        start_pos = (i * chunk_size) + (random.randint(0, max_start) if max_start > 0 else 0)
        end_pos   = start_pos + len(snippet)

        # Add the snippet to the target audio (this represents talking over each other)
        # Ensure we don't go out of bounds
        if end_pos <= len_t:
            y_mix[start_pos:end_pos] += snippet
        else:
            fit_length = len_t - start_pos
            y_mix[start_pos:] += snippet[:fit_length]

    # Prevent clipping
    max_val = np.max(np.abs(y_mix))
    if max_val >= 1.0:
        y_mix = y_mix * (0.9 / max_val) # Peak normalize to 0.9 to prevent digital distortion

    if output_path is not None:
        # Save file
        sf.write(output_path, y_mix, TARGET_SR)

    return y_mix

In [21]:
# Preview Function
def preview_mixes(df, n=3, audio_cache=None, base_path=None, random_state=42):
    """
    Preview n random mixes — plays target, scaled noise, and mixed audio.
    For file-based audio (LibriSpeech): pass base_path.
    For cache-based audio (WAXAL): pass audio_cache.
    """
    # Extract mixture metadata from the current row
    sample = df.sample(n=n, random_state=random_state).reset_index(drop=True)

    for i, row in sample.iterrows():
        sir_db, overlap, mix_id = row['sir_level_db'], row['overlap_ratio'], row['mix_id']

        # For Twi dataset, fetch raw audio arrays from the in-memory cache
        if audio_cache is not None:
            y_t = audio_cache.get(row['target_audio_path'])
            y_n = audio_cache.get(row['noise_audio_path'])

            # Validation check to ensure both target and noise are present in memory
            if y_t is None or y_n is None:
                print(f"Audio not found in cache for {mix_id}, skipping.")
                continue
        # For English dataset, load files from disk
        else:
            t_path = row['target_audio_path']
            n_path = row['noise_audio_path']

            # Handle relative paths by joining them with the project's base directory
            if base_path:
                t_path = t_path if os.path.isabs(t_path) else os.path.join(base_path, t_path)
                n_path = n_path if os.path.isabs(n_path) else os.path.join(base_path, n_path)
            y_t = preprocess_audio(t_path).squeeze(0).numpy()
            y_n = preprocess_audio(n_path).squeeze(0).numpy()

        y_n_scaled = apply_energy_scaling(y_t, y_n, sir_db)

        # Mix audios without saving
        y_mix = mix_audio_arrays(y_t, y_n, sir_db, overlap, output_path=None)

        print(f"{'-'*60}")
        print(f"Mix {i+1}: {mix_id}  |  SIR={sir_db} dB  |  Overlap={overlap:.0%}")
        print(f"{'-'*60}")
        print("Target speech")
        display(ipd.Audio(y_t, rate=TARGET_SR))
        print("Interfering speech (scaled to SIR)")
        display(ipd.Audio(y_n_scaled, rate=TARGET_SR))
        print("Mixed audio")
        display(ipd.Audio(y_mix, rate=TARGET_SR))
        print()

In [22]:
# Mixing Loop
def run_mixing_loop(df, output_dir, audio_cache=None, base_path=None):
    """
    Generate and save mixed audio for all rows in df.
    Skips files that already exist on disk.
    For LibriSpeech: pass base_path to resolve relative audio paths.
    For WAXAL: pass audio_cache dict.
    """
    skipped, errors = 0, 0

    for _, row in tqdm(df.iterrows(), total=df.shape[0]):
        sir, overlap, mix_id = row['sir_level_db'], row['overlap_ratio'], row['mix_id']
        output_path = os.path.join(output_dir, f"{mix_id}_sir{sir}_ov{overlap}.wav")

        if os.path.exists(output_path):
            skipped += 1
            continue

        try:
            # For Twi dataset, fetch raw audio arrays from the in-memory cache
            if audio_cache is not None:
                y_t = audio_cache.get(row['target_audio_path'])
                y_n = audio_cache.get(row['noise_audio_path'])

                # Validation check to ensure both target and noise are present in memory
                if y_t is None or y_n is None:
                    print(f"Skipping {mix_id}: not found in cache.")
                    errors += 1
                    continue
                mix_audio_arrays(y_t, y_n, sir, overlap, output_path)
            # For English dataset, load files from disk
            else:
                t_path = row['target_audio_path']
                n_path = row['noise_audio_path']

                # Handle relative paths by joining them with the project's base directory
                if base_path:
                    t_path = t_path if os.path.isabs(t_path) else os.path.join(base_path, t_path)
                    n_path = n_path if os.path.isabs(n_path) else os.path.join(base_path, n_path)
                mix_audio_arrays(
                    preprocess_audio(t_path).squeeze(0).numpy(),
                    preprocess_audio(n_path).squeeze(0).numpy(),
                    sir, overlap, output_path
                )
        except Exception as e:
            print(f"Error processing {mix_id}: {e}")
            errors += 1

    print(f"\nDone. Skipped (already exist): {skipped} | Errors: {errors}")

In [23]:
# English - Load CSV & Preview
df_eng = pd.read_csv(ENG_CSV)
print(f"Loaded {len(df_eng)} LibriSpeech recipes.\n")

preview_mixes(df_eng, n=3, base_path=BASE_PATH)

Loaded 12500 LibriSpeech recipes.

------------------------------------------------------------
Mix 1: mix_librispeech_1766  |  SIR=5 dB  |  Overlap=20%
------------------------------------------------------------
Target speech


Interfering speech (scaled to SIR)


Mixed audio



------------------------------------------------------------
Mix 2: mix_librispeech_11919  |  SIR=-5 dB  |  Overlap=40%
------------------------------------------------------------
Target speech


Interfering speech (scaled to SIR)


Mixed audio



------------------------------------------------------------
Mix 3: mix_librispeech_8909  |  SIR=-5 dB  |  Overlap=10%
------------------------------------------------------------
Target speech


Interfering speech (scaled to SIR)


Mixed audio


In [10]:
# English - Run Mixing Loop
run_mixing_loop(df_eng, ENG_OUTPUT_DIR, base_path=BASE_PATH)
print("LibriSpeech mixing complete!")

100%|██████████| 12500/12500 [23:57<00:00,  8.69it/s]


Done. Skipped (already exist): 177 | Errors: 0
LibriSpeech mixing complete!


## Twi: WAXAL Dataset Audio Processing

In [11]:
# Twi - Build Audio Cache
df_twi = pd.read_csv(TWI_CSV)
print(f"Loaded {len(df_twi)} WAXAL recipes.")

Loaded 12500 WAXAL recipes.


In [12]:
# Build in-memory cache of WAXAL data
waxal_audio_cache = build_waxal_cache(WAXAL_DISK_PATH)

Loading WAXAL dataset from /content/drive/MyDrive/Datasets/waxalnlp_aka_asr_disk...


Loading dataset from disk:   0%|          | 0/109 [00:00<?, ?it/s]

Building cache: 100%|██████████| 1522/1522 [01:27<00:00, 17.34it/s]

Cached 1522 audio arrays.


In [13]:
# Twi - Preview
preview_mixes(df_twi, n=3, audio_cache=waxal_audio_cache)

------------------------------------------------------------
Mix 1: mix_waxal_1766  |  SIR=-5 dB  |  Overlap=10%
------------------------------------------------------------
Target speech


Interfering speech (scaled to SIR)


Mixed audio



------------------------------------------------------------
Mix 2: mix_waxal_11919  |  SIR=-5 dB  |  Overlap=50%
------------------------------------------------------------
Target speech


Interfering speech (scaled to SIR)


Mixed audio



------------------------------------------------------------
Mix 3: mix_waxal_8909  |  SIR=-5 dB  |  Overlap=20%
------------------------------------------------------------
Target speech


Interfering speech (scaled to SIR)


Mixed audio


In [14]:
# Twi - Run Mixing Loop
run_mixing_loop(df_twi, TWI_OUTPUT_DIR, audio_cache=waxal_audio_cache)
print("WAXAL mixing complete!")

100%|██████████| 12500/12500 [10:55<00:00, 19.07it/s]


Done. Skipped (already exist): 0 | Errors: 0
WAXAL mixing complete!
